# Charge threshold DAC to photoelectron conversion

For each charge-threshold DAC setting, locate the ADC bin at which the
threshold cuts the spectrum, then convert that position into
photoelectrons using a gain calibration.

**Pipeline**

`discover threshold folders -> read channel -> locate leading edge ->
fit DAC vs ADC -> apply gain calibration -> DAC vs p.e.`

Two edge-finding methods are used. The simple one takes the first bin
above a noise floor. The erfc one fits the rise of the cumulative
spectrum.

Author: Muhammad Abdullahi (muhammad.abdullahi@gssi.it)


## Imports


In [ ]:
import os
import re
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
from scipy.ndimage import gaussian_filter1d
from scipy.optimize import curve_fit
from scipy.special import erfc
from scipy.signal import find_peaks


## Configuration

Paths are relative to this notebook. Point `_pwd` at the folder holding
the threshold-scan acquisitions. Nothing below refers to an absolute path.

Each acquisition folder is expected to carry its DAC value in its name, for
example `Fine_HLL_38V_HG=50_Chargeth_240/`, and to contain `data.csv`.


In [ ]:
_pwd       = './data/threshold_scan/'   # acquisition folders live here
plt_folder = './Results'                # figures are written here
csv_name   = 'data.csv'

# Channel under test
DAQ, ASIC, CHANNEL, GAIN = 1, 'A', 1, 'HG'

# Gain calibration.  If the JSON written by HG_gain_calibration.ipynb is
# present it is used.  Otherwise the gain is derived from the lowest
# threshold spectrum, which is less reliable (see the calibration section).
gain_json = os.path.join(plt_folder,
                         f'spacing_DAQ{DAQ}_{ASIC}_CH{CHANNEL}_{GAIN}.json')

# Peak finding, used only when deriving the gain locally
SIGMA, PROM_FRAC, MIN_DIST, MAX_PE = 4.0, 0.04, 8, 5

# Leading-edge detection
SMOOTH_SIGMA = 3.0    # gaussian smoothing before edge finding
NOISE_FRAC   = 0.01   # fraction of maximum, for the simple method

savefig = True
os.makedirs(plt_folder, exist_ok=True)


## Channel mapping

| range | contents |
|---|---|
| 0-159   | DAQ1, ASIC A..E, LG, CH0..31 |
| 160-319 | DAQ1, ASIC A..E, HG, CH0..31 |
| 320-479 | DAQ2, ASIC A..E, LG, CH0..31 |
| 480-639 | DAQ2, ASIC A..E, HG, CH0..31 |


In [ ]:
ASIC_ORDER = ['A', 'B', 'C', 'D', 'E']


def col_index(daq, asic, ch, gain):
    if daq not in (1, 2):
        raise ValueError('daq must be 1 or 2')
    asic = asic.upper()
    if asic not in ASIC_ORDER:
        raise ValueError(f'asic must be one of {ASIC_ORDER}')
    if not (0 <= ch <= 31):
        raise ValueError('ch must be 0..31')
    gain = gain.upper()
    if gain not in ('LG', 'HG'):
        raise ValueError("gain must be 'LG' or 'HG'")
    return ((daq - 1) * 320
            + (0 if gain == 'LG' else 160)
            + ASIC_ORDER.index(asic) * 32
            + ch)


def load_channel(csv_path, col):
    """One column of the histogram CSV."""
    return (pd.read_csv(csv_path, header=None, usecols=[col])
              .iloc[:, 0].to_numpy(dtype=float))


col = col_index(DAQ, ASIC, CHANNEL, GAIN)
print(f'DAQ{DAQ} ASIC-{ASIC} CH{CHANNEL} {GAIN} is column {col}')


## File discovery

The DAC value is read from each folder name rather than assumed from list
position, so adding or reordering acquisitions cannot mislabel them.


In [ ]:
def discover_threshold_scan(folder, csv_name=csv_name,
                            pattern=r'(?:Chargeth|threshold)[_\- ]?(\d+)'):
    """Map DAC value -> path of its CSV."""
    if not os.path.isdir(folder):
        raise FileNotFoundError(f'Data folder not found: {folder}')

    found = {}
    for root, _dirs, files in os.walk(folder):
        if csv_name not in files:
            continue
        m = re.search(pattern, os.path.basename(root), re.IGNORECASE)
        if not m:
            print(f'  skipped, no DAC value in name: {os.path.basename(root)}')
            continue
        dac = int(m.group(1))
        path = os.path.join(root, csv_name)
        if dac in found:
            raise ValueError(f'DAC={dac} appears twice: '
                             f'{found[dac]} and {path}')
        found[dac] = path
    return found


csv_path_by_dac = discover_threshold_scan(_pwd)
dac_values = sorted(csv_path_by_dac)
print(f'{len(dac_values)} thresholds found: {dac_values}')


## Load the spectra


In [ ]:
spectra = {dac: load_channel(csv_path_by_dac[dac], col)
           for dac in dac_values}
print(f'loaded {len(spectra)} spectra, {len(next(iter(spectra.values())))} bins each')


## Gain calibration

The conversion needs `ADC = gain x p.e. + pedestal`.

If `HG_gain_calibration.ipynb` has been run for this channel its JSON is
used directly. Otherwise the gain is derived here from the lowest-threshold
spectrum, which carries an assumption worth stating: the peaks found are
numbered 1, 2, 3 upward, so the first visible peak is taken to be one
photoelectron. In a triggered spectrum the threshold has already removed
the pedestal and possibly the first photoelectron peaks, in which case that
numbering is wrong and both the gain and the pedestal shift. Prefer the
external calibration where one exists.


In [ ]:
def gain_from_json(path, hg_code=None):
    """Read gain from the HG calibration notebook output, if present."""
    if not os.path.isfile(path):
        return None
    with open(path) as f:
        s = json.load(f)
    res = s.get('results', {})
    if not res:
        return None
    key = str(hg_code) if hg_code is not None else sorted(res)[0]
    if key not in res:
        return None
    return float(res[key]['spacing']), float(res[key]['pedestal']), key


def gain_from_spectrum(y, sigma=SIGMA, prom_frac=PROM_FRAC,
                       min_dist=MIN_DIST, max_pe=MAX_PE):
    """Fallback: fit ADC against peak order in a reference spectrum."""
    ys = gaussian_filter1d(y, sigma=sigma)
    peaks, _ = find_peaks(
        ys, prominence=prom_frac * ys.max() if ys.max() > 0 else 0,
        distance=min_dist)
    if len(peaks) < 2:
        raise RuntimeError(
            f'Cannot fit gain: found {len(peaks)} peaks, need at least 2. '
            'Loosen PROM_FRAC or MIN_DIST, or supply an external '
            'calibration.')
    n = min(len(peaks), max_pe)
    adc = peaks[:n].astype(float)
    pe = np.arange(1, n + 1, dtype=float)
    gain, pedestal = np.polyfit(pe, adc, 1)
    return float(gain), float(pedestal), pe, adc


cal = gain_from_json(gain_json)
if cal is not None:
    REF_GAIN, REF_PEDESTAL, src = cal
    print(f'Gain from {gain_json} (HG={src})')
else:
    REF_GAIN, REF_PEDESTAL, cal_pe, cal_adc = gain_from_spectrum(
        spectra[dac_values[0]])
    print(f'Gain derived locally from the DAC={dac_values[0]} spectrum')

print(f'  ADC = {REF_GAIN:.2f} x p.e. + {REF_PEDESTAL:.1f}')


## Locating the threshold edge

`erfc_rise` is a cumulative distribution running from `baseline` to
`baseline + amp`, so fitting normalised data gives `baseline` near 0 and
`amp` near 1 and the fitted `adc_mid` is the true 50% point.

Note what that 50% point means. The cumulative of the recorded spectrum
rises across the whole ADC range, not only at the threshold, so its median
tracks the threshold but is displaced by the shape of the surviving
spectrum. That displacement changes with DAC, so it can bias the slope of
the DAC to ADC fit. The simple method is free of this effect but is more
sensitive to noise. Compare the two before trusting either.


In [ ]:
def erfc_rise(adc, adc_mid, sigma, amp, baseline):
    """Cumulative rise from `baseline` to `baseline + amp`."""
    z = -(adc - adc_mid) / (np.sqrt(2.0) * max(sigma, 0.1))
    return baseline + amp * 0.5 * erfc(z)


def threshold_adc_simple(y, noise_frac=NOISE_FRAC,
                         smooth_sigma=SMOOTH_SIGMA):
    """First bin whose smoothed content exceeds a fraction of the max."""
    ys = gaussian_filter1d(y, sigma=smooth_sigma)
    if ys.max() <= 0:
        return None
    above = np.where(ys > noise_frac * ys.max())[0]
    return int(above[0]) if len(above) else None


def threshold_adc_erfc(y, smooth_sigma=SMOOTH_SIGMA):
    """50% point of the cumulative spectrum, from an erfc fit."""
    ys = gaussian_filter1d(y, sigma=smooth_sigma)
    total = ys.sum()
    if total <= 0:
        return None, None

    cum = np.cumsum(ys) / total
    rising = np.where((cum > 0.001) & (cum < 0.99))[0]
    if len(rising) < 5:
        return None, None

    lo = max(0, rising[0] - 20)
    hi = min(len(cum), rising[-1] + 20)
    x_fit = np.arange(lo, hi, dtype=float)
    y_fit = cum[lo:hi]

    p0 = [x_fit[np.argmin(np.abs(y_fit - 0.5))], 10.0, 1.0, 0.0]
    bounds = ([x_fit[0], 1.0, 0.5, -0.1],
              [x_fit[-1], 200.0, 1.5, 0.1])
    try:
        popt, _ = curve_fit(erfc_rise, x_fit, y_fit, p0=p0,
                            bounds=bounds, maxfev=20000)
    except (RuntimeError, ValueError):
        return None, None
    return float(popt[0]), float(popt[1])


## Extract the edge for every threshold


In [ ]:
rows = []
for dac in dac_values:
    y = spectra[dac]
    a_s = threshold_adc_simple(y)
    a_e, sig = threshold_adc_erfc(y)
    rows.append({
        'dac': dac, 'adc_simple': a_s, 'adc_erfc': a_e, 'sigma': sig,
        'pe_simple': (a_s - REF_PEDESTAL) / REF_GAIN if a_s is not None else None,
        'pe_erfc':   (a_e - REF_PEDESTAL) / REF_GAIN if a_e is not None else None,
    })

table = pd.DataFrame(rows).set_index('dac')
print(f'Channel: DAQ{DAQ} ASIC-{ASIC} CH{CHANNEL} {GAIN} (column {col})')
print(f'Gain: ADC = {REF_GAIN:.2f} x p.e. + {REF_PEDESTAL:.1f}')
print()
print(table.round(2).to_string())


## DAC to ADC, and the full chain to photoelectrons

The erfc estimate is used where the fit converged, otherwise the simple
one.


In [ ]:
dac_arr = table.index.to_numpy(dtype=float)
adc_arr = np.array([e if e is not None else (s if s is not None else np.nan)
                    for e, s in zip(table['adc_erfc'], table['adc_simple'])],
                   dtype=float)
valid = ~np.isnan(adc_arr)

if valid.sum() < 2:
    raise RuntimeError('Fewer than two usable points, cannot fit.')

coef, cov = np.polyfit(dac_arr[valid], adc_arr[valid], 1, cov=True)
slope_da, offset_da = coef
slope_err = float(np.sqrt(cov[0, 0]))

pe_slope = slope_da / REF_GAIN
pe_offset = (offset_da - REF_PEDESTAL) / REF_GAIN

print(f'DAC -> ADC : ADC = {slope_da:.3f} (+/- {slope_err:.3f}) x DAC '
      f'+ {offset_da:.1f}')
print(f'             1 DAC step = {slope_da:.3f} ADC')
print()
print(f'Full chain : p.e. = {pe_slope:.4f} x DAC + ({pe_offset:.2f})')
print(f'  inverse  : DAC  = {1 / pe_slope:.1f} x p.e. '
      f'+ ({-pe_offset / pe_slope:.1f})')
print()
for d in dac_values[::max(1, len(dac_values) // 5)]:
    a = slope_da * d + offset_da
    print(f'  DAC {d:>4} -> ADC {a:7.0f} -> {(a - REF_PEDESTAL) / REF_GAIN:6.2f} p.e.')


## Spectra with the located thresholds


In [ ]:
def plot_spectra(spectra, table, save=savefig):
    n = len(spectra)
    colors = cm.cool(np.linspace(0, 1, n))
    fig, ax = plt.subplots(figsize=(12, 7))

    stack = np.column_stack(list(spectra.values()))
    nz = np.where(stack.max(axis=1) > 0)[0]
    lo, hi = (max(0, nz[0] - 30), min(stack.shape[0], nz[-1] + 50)) \
        if len(nz) else (0, stack.shape[0])

    for i, dac in enumerate(sorted(spectra)):
        ys = gaussian_filter1d(spectra[dac], sigma=SMOOTH_SIGMA)
        ax.plot(ys, color=colors[i], lw=1.2, alpha=0.8, label=f'DAC {dac}')
        pos = table.loc[dac, 'adc_erfc']
        if pos is None or (isinstance(pos, float) and np.isnan(pos)):
            pos = table.loc[dac, 'adc_simple']
        if pos is not None and not (isinstance(pos, float) and np.isnan(pos)):
            ax.axvline(pos, color=colors[i], ls='--', alpha=0.4, lw=1)

    ax.set_xlim(lo, hi)
    ax.set_yscale('log')
    ax.set_ylim(0.5, None)
    ax.set_xlabel('ADC bin', fontsize=13)
    ax.set_ylabel('Counts (smoothed)', fontsize=13)
    ax.set_title(f'Spectra with threshold positions, DAQ{DAQ} ASIC-{ASIC} '
                 f'CH{CHANNEL} {GAIN}', fontsize=14)
    ax.legend(fontsize=7, ncol=3, loc='upper right')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    if save:
        p = os.path.join(plt_folder,
                         f'spectra_DAQ{DAQ}_{ASIC}_CH{CHANNEL}_{GAIN}.pdf')
        plt.savefig(p, dpi=300)
        print(f'Plot saved: {p}')
    plt.show()
    plt.close()


plot_spectra(spectra, table)


## DAC against ADC


In [ ]:
def plot_dac_vs_adc(table, save=savefig):
    fig, ax = plt.subplots(figsize=(8, 6))
    d = table.index.to_numpy(dtype=float)

    m = table['adc_simple'].notna()
    ax.plot(d[m.to_numpy()], table.loc[m, 'adc_simple'], 'o',
            color='steelblue', ms=8, alpha=0.5,
            label='simple (first bin above noise)')
    m = table['adc_erfc'].notna()
    ax.plot(d[m.to_numpy()], table.loc[m, 'adc_erfc'], 'D',
            color='darkorange', ms=9, label='erfc fit (50% point)')

    grid = np.linspace(d.min() - 10, d.max() + 10, 100)
    ax.plot(grid, slope_da * grid + offset_da, '-', color='red', lw=2.5,
            label=f'ADC = {slope_da:.2f} x DAC + {offset_da:.1f}')

    ax.set_xlabel('Charge threshold DAC', fontsize=14)
    ax.set_ylabel('Threshold position (ADC bin)', fontsize=14)
    ax.set_title(f'DAC to ADC, DAQ{DAQ} ASIC-{ASIC} CH{CHANNEL} {GAIN}',
                 fontsize=14)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    if save:
        p = os.path.join(plt_folder,
                         f'dac_vs_adc_DAQ{DAQ}_{ASIC}_CH{CHANNEL}_{GAIN}.pdf')
        plt.savefig(p, dpi=300)
        print(f'Plot saved: {p}')
    plt.show()
    plt.close()


plot_dac_vs_adc(table)


## DAC against photoelectrons


In [ ]:
def plot_dac_vs_pe(table, save=savefig):
    fig, ax = plt.subplots(figsize=(8, 6))
    d = table.index.to_numpy(dtype=float)[valid]
    pe = (adc_arr[valid] - REF_PEDESTAL) / REF_GAIN

    ax.plot(d, pe, 'D', color='steelblue', ms=10, zorder=5,
            label='from DAC to ADC to p.e.')
    grid = np.linspace(d.min() - 10, d.max() + 10, 100)
    ax.plot(grid, (slope_da * grid + offset_da - REF_PEDESTAL) / REF_GAIN,
            '-', color='red', lw=2.5,
            label=f'p.e. = {pe_slope:.4f} x DAC + ({pe_offset:.2f})')

    ax.set_xlabel('Charge threshold DAC', fontsize=14)
    ax.set_ylabel('Threshold (p.e.)', fontsize=14)
    ax.set_title(f'DAC to p.e., DAQ{DAQ} ASIC-{ASIC} CH{CHANNEL} {GAIN}',
                 fontsize=14)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    if save:
        p = os.path.join(plt_folder,
                         f'dac_vs_pe_DAQ{DAQ}_{ASIC}_CH{CHANNEL}_{GAIN}.pdf')
        plt.savefig(p, dpi=300)
        print(f'Plot saved: {p}')
    plt.show()
    plt.close()


plot_dac_vs_pe(table)


## Save the numbers


In [ ]:
summary = {
    'daq': DAQ, 'asic': ASIC, 'channel': CHANNEL, 'gain': GAIN,
    'gain_calibration': {'gain': REF_GAIN, 'pedestal': REF_PEDESTAL},
    'dac_to_adc': {'slope': float(slope_da), 'slope_err': slope_err,
                   'offset': float(offset_da)},
    'dac_to_pe': {'slope': float(pe_slope), 'offset': float(pe_offset)},
    'per_threshold': table.where(pd.notna(table), None).to_dict('index'),
}

out = os.path.join(plt_folder,
                   f'dac_to_pe_DAQ{DAQ}_{ASIC}_CH{CHANNEL}_{GAIN}.json')
with open(out, 'w') as f:
    json.dump(summary, f, indent=2, default=float)
print(f'Results saved: {out}')

csv_out = out.replace('.json', '.csv')
table.to_csv(csv_out)
print(f'Table saved:   {csv_out}')
